In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [2]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "../hf_home"  # stores logins

hf_token = os.getenv('HF_TOKEN')
# print(type(hf_token))
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path

variants = ['small', 'base', 'large']

paths = {variant:f'../models/google/flan-t5-{variant}' for variant in variants}

model_var = 'small'
current_path =paths[model_var]
save_path = Path(current_path).resolve()

tokenizer = AutoTokenizer.from_pretrained(
    save_path,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    save_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

print("Reloaded model successfully")
print(f"model.device = {model.device}")

Reloaded model successfully
model.device = mps:0


#Dataloading (TODO)
#after sparqlgen + name parsing

In [37]:
import pandas as pd

In [38]:
names = ['mintaka', 'hotpot', 'qald']

dataframes = {name:pd.read_csv(f'../data/processed_names/{name}_processed.csv') for name in names}





In [39]:
# from transformers import pipeline
from transformers.generation.utils import GenerationMixin

In [ ]:
def answer(question, context):
    # Simpler, more direct prompt for FLAN
    prompt = f"""Context: {context}

Question: {question}

Answer the question ONLY using the context provided, don't rely on your own knowledge. Only make claims that can be supported by context."""
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        temperature = 0,
        early_stopping=True,
        do_sample=False,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    return generated_text

answered = {}
for name, data in dataframes.items():
    print(f'Starting processing for {name}')
    answers = []
    for index, row in data.iterrows():
        question = row['SAE Question']
        context = row['names']
        llmanswer = answer(question, context)
        answers.append(llmanswer)
        print(f'Row {index+1} processed', llmanswer)
    data['Answer'] = answers
    answered[name] = data

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Starting processing for mintaka
Row 1 processed I don't know
Row 2 processed I don't know
Row 3 processed I don't know
Row 4 processed I don't know
Row 5 processed Donald Trump.
Row 6 processed I don't know
Row 7 processed I don't know
Row 8 processed I do not know
Row 9 processed I don't know
Row 10 processed I don't know
Row 11 processed I don't know
Row 12 processed I don't know
Row 13 processed California.
Row 14 processed I don't know
Row 15 processed I don't know
Row 16 processed Staten Island
Row 17 processed I don't know
Row 18 processed Super Mario Bros
Row 19 processed Wario Land.
Row 20 processed I don't know
Row 21 processed 0
Row 22 processed I don't know
Row 23 processed Michael Phelps
Row 24 processed No
Row 25 processed I don't know
Row 26 processed No
Row 27 processed No
Row 28 processed I don't know
Row 29 processed Yes
Row 30 processed I don't know
Row 31 processed I don't know
Row 32 processed Dwayne Johnson
Row 33 processed Yes.
Row 34 processed Joker
Row 35 proces

In [ ]:
import ast
import re

def clean_text(text):
    # If the entry is actually a list, flatten it
    if isinstance(text, list):
        text = text[0]
    elif isinstance(text, str):
       #If looks like a list []
        if text.strip().startswith('[') and text.strip().endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list) and len(parsed) > 0:
                    text = parsed[0]
            except Exception:
                pass

    # Now clean as before
    if not isinstance(text, str):
        return text

    text = re.sub(r'^(AAVE|SAE)\s*Question[:\s-]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\s\[\]\'"]+|[\s\[\]\'"]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



for name, data in answered.items():
# Apply to both columns
  data['SAE Question'] = data['SAE Question'].apply(clean_text)
  data['AAVE Question'] = data['AAVE Question'].apply(clean_text)
  file_path = f'../data/llm_answers/flan-t5-{model_var}/LLM_Answers_{name}.csv'
  data.to_csv(file_path, index=False)
  print(f"Translations completed and saved to {file_path}")





Translations completed and saved to ./LLM_answers/flan-t5-large/LLM_Answers_mintaka.csv
Translations completed and saved to ./LLM_answers/flan-t5-large/LLM_Answers_hotpot.csv
Translations completed and saved to ./LLM_answers/flan-t5-large/LLM_Answers_qald.csv


: 